[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/begelb/latent_dynamics/blob/paper/notebooks/03_leslie3d_example1.ipynb)

In [ ]:
# Install CMGDB and the paper repository (the clone carries the saved model
# weights the notebooks recompute from). Running locally inside the repo,
# skip this cell.
!git clone -q --depth 1 --branch paper https://github.com/begelb/latent_dynamics.git
!pip install -q CMGDB
!pip install -q -e latent_dynamics
%cd latent_dynamics

# Section 5.2.1 - Three-dimensional Leslie: a spurious attractor

## What this notebook shows

The three-generation **Leslie population model** (paper section 5.2), a
genuinely three-dimensional system, studied through a **two-dimensional** latent
model. This is the *cautionary tale*: the latent Morse graph comes out
**tristable** (three minimal nodes), but the true system has only two stable
period-four orbits -- one latent attractor corresponds to no attractor of the
real dynamics.

This is not just numerical noise: the semiconjugacy-error bound required by the
main theorem is *violated* at that node. A small training loss does **not** by
itself guarantee that latent attractors are real. (Example 2, in the next
notebook, is the success case.)

### How to run

Set `MODE` in the parameters cell, then Run All.

| `MODE` | what it does | typical cost |
|--------|--------------|--------------|
| `"quick"` | recompute the Morse graph of the *saved* model on the coarse `QUICK_SUBDIV` grid | seconds |
| `"morse"` | recompute the Morse graph of the *saved* model at your `SUBDIV` (paper value by default) | minutes |
| `"retrain"` | train a fresh model, then compute its Morse graph at your `SUBDIV` | minutes-hours (GPU recommended) |

**Coarse grids can merge nearby recurrent sets and change the Morse graph**,
so `quick` is a preview, not a paper-quality result. Nothing a notebook does
ever touches the preserved paper trees: recomputes land under
`output/notebooks/<experiment>/`.

In [ ]:
# ===== PARAMETERS  (edit, then Run All) ====================================
MODE = "quick"             # "quick" | "morse" | "retrain"
SEED = None                # None -> the config's default seed
QUICK_SUBDIV = (10, 14, 20)  # MODE="quick": coarse preview grid, runs in seconds
SUBDIV = (23, 23, 27)      # MODE="morse"/"retrain": (subdiv_init, subdiv_min, subdiv_max)
                           # the paper value
BOX_SCALE = "auto"         # Morse-set box size: "auto" | float | {label: float}
# ===========================================================================

REPLAY_CONFIG  = "leslie3d_example1_replay"
RETRAIN_CONFIG = "leslie3d_example1"

## The system

The three-generation Leslie map (paper section 5.2)

$$f(x) = \big((\theta_1 x_1 + \theta_2 x_2 + \theta_3 x_3)\,e^{-0.1(x_1+x_2+x_3)},\;
p_1 x_1,\; p_2 x_2\big)$$

on the positive octant. The map is only three-dimensional, so nothing is
embedded here: the question is what a *two*-dimensional latent model makes of
it, and this example is the one where an extra, spurious attractor appears.

In [ ]:
# ---- the system (paper values; edit and re-run to explore) ----------------
TH1, TH2, TH3 = 28.9, 29.8, 22.0    # fecundities
SURVIVAL_1, SURVIVAL_2 = 0.7, 0.7   # survival between generations
LATENT_DIMS = 2

from latentdynamics.config import load_config
from latentdynamics.systems import build_system

SYSTEM_PARAMS = {
    "th1": TH1, "th2": TH2, "th3": TH3,
    "survival_p1": SURVIVAL_1, "survival_p2": SURVIVAL_2,
}
system = build_system("leslie3d", SYSTEM_PARAMS)
print(f"ambient dimension {system.dim}, latent dimension {LATENT_DIMS}")
print(f"phase space  lower {system.lower_bounds.tolist()}")
print(f"             upper {system.upper_bounds.tolist()}")

## The autoencoder and its latent map

An encoder, a decoder, and a latent map trained together so the latent map is
an $\epsilon$-approximate semiconjugacy to the full system on the data.

In [ ]:
# ---- the autoencoder (paper values) --------------------------------------
HIDDEN_SHAPES = [32, 32, 32]        # per component: encoder, latent map, decoder
LOSS_WEIGHTS = [10.0, 10.0, 1.0]    # (w1, w2, w3): reconstruction, latent step, cycle
LEARNING_RATE = 1e-3
BATCH_SIZE = 1024
EPOCHS = 1000
PATIENCE = 100

print(f"{system.dim} -> {LATENT_DIMS} -> {system.dim}, hidden {HIDDEN_SHAPES} per component")
print(f"loss weights {LOSS_WEIGHTS}, Adam lr {LEARNING_RATE}, batch {BATCH_SIZE}")

## The data

Pairs $(x, f(x))$ along trajectories from sampled initial conditions: the model
only ever sees one-step transitions, never the map itself.

In [ ]:
# ---- the data (paper values) ---------------------------------------------
N_TRAIN = 4000        # training trajectories
N_VAL = 5000          # validation trajectories
N_ITERATIONS = 30     # steps per trajectory
SKIP = 10             # transient steps discarded before recording pairs

print(f"{N_TRAIN} train / {N_VAL} validation trajectories, "
      f"{N_ITERATIONS} steps each, first {SKIP} discarded")

# Everything above is fed to the pipeline as config overrides, so `retrain`
# below trains exactly the model described here.
OVERRIDES = {
    "system": {"params": SYSTEM_PARAMS},
    "arch": {
        "low_dims": LATENT_DIMS,
        "encoder": {"hidden_shapes": HIDDEN_SHAPES},
        "latent_map": {"hidden_shapes": HIDDEN_SHAPES},
        "decoder": {"hidden_shapes": HIDDEN_SHAPES},
    },
    "training": {
        "loss_weights": LOSS_WEIGHTS,
        "learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "patience": PATIENCE,
    },
    "data": {
        "n_samples_val": N_VAL,
        "n_iterations": N_ITERATIONS,
        "skip": SKIP,
    },
}

# Flag anything that no longer matches the paper's configuration.
paper = load_config(RETRAIN_CONFIG)
drift = []
if SYSTEM_PARAMS != paper.system.params:
    drift.append(f"system {paper.system.params}")
if LATENT_DIMS != paper.arch.low_dims:
    drift.append(f"latent dims {paper.arch.low_dims}")
if HIDDEN_SHAPES != paper.arch.encoder.hidden_shapes:
    drift.append(f"encoder hidden {paper.arch.encoder.hidden_shapes}")
for name, value, reference in [
    ("loss weights", LOSS_WEIGHTS, paper.training.loss_weights),
    ("learning rate", LEARNING_RATE, paper.training.learning_rate),
    ("batch size", BATCH_SIZE, paper.training.batch_size),
    ("epochs", EPOCHS, paper.training.epochs),
    ("patience", PATIENCE, paper.training.patience),
    ("validation trajectories", N_VAL, paper.data.n_samples_val),
    ("iterations", N_ITERATIONS, paper.data.n_iterations),
]:
    if value != reference:
        drift.append(f"{name} {reference}")
print("\nmatches the paper's configuration" if not drift
      else "\ndiffers from the paper, which uses: " + "; ".join(drift))

## Training

`quick` and `morse` load the paper's trained weights. `retrain` runs the data,
scaling, training and diagnostic stages and then CMGDB itself, at the
`SUBDIV` set above.

In [ ]:
from latentdynamics.replay import load_experiment, retrain

if MODE in ("quick", "morse"):
    exp = load_experiment(REPLAY_CONFIG, seed=SEED)
elif MODE == "retrain":
    # The retrain pipeline runs CMGDB itself, at the SUBDIV set above.
    OVERRIDES["cmgdb"] = {
        "subdiv_init": SUBDIV[0],
        "subdiv_min": SUBDIV[1],
        "subdiv_max": SUBDIV[2],
    }
    exp = retrain(
        RETRAIN_CONFIG,
        seed=SEED,
        overrides=OVERRIDES,
        stages=("data", "scale", "train", "diagnose", "morse"),
    )
else:
    raise ValueError(f"unknown MODE {MODE!r}")
exp

## Training curves

Total loss and its terms, per epoch. In `quick` and `morse` mode these are the
saved curves of the run the paper reports; in `retrain` mode they are the run
that just finished. Training stops early when validation loss stalls, so the
curves usually end well before the configured epoch budget.

In [ ]:
import json

import matplotlib.pyplot as plt

history_path = exp.seed_dir / "logs" / "history.json"
if not history_path.exists():
    print(f"no training history at {history_path}")
else:
    history = json.loads(history_path.read_text())
    terms = [k for k in history["train"] if k != "loss_total"]

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
    left.semilogy(history["train"]["loss_total"], label="train")
    left.semilogy(history["val"]["loss_total"], label="validation")
    left.set(xlabel="epoch", ylabel="total loss")
    left.legend()

    for term in terms:
        right.semilogy(history["val"][term], label=term)
    right.set(xlabel="epoch", ylabel="validation loss by term")
    right.legend(fontsize="small")

    fig.tight_layout()
    plt.show()
    print(f"{len(history['train']['loss_total'])} epochs")

## Morse graph

CMGDB subdivides the latent rectangle into boxes and builds the directed graph
on them induced by the latent map: box `B` points at every box meeting an
enclosure of `g(B)`. The **Morse sets** are that graph's strongly connected
components, and the **Morse graph** is its condensation, with a Conley index on
each node. `ComputeConleyMorseGraph` returns both -- the map graph is not an
extra step, it *is* the computation.

The grid comes from `QUICK_SUBDIV` in `quick` mode and from `SUBDIV`
otherwise, chosen here rather than inherited from the training config.

The box map evaluates the network on the corner lattice in batches -- one call
per batch instead of one per box. CMGDB (>= 1.5.0) caches the transition graph in one block with
automatic edge reservation, so no environment tuning is needed; deep grids
are limited by memory alone.

In [ ]:
# quick and morse recompute the Morse graph of the saved model; retrain already
# computed it during the pipeline run above. Either way the artifacts live in
# the notebook playground, never in the preserved paper trees.
if MODE == "retrain":
    run = exp
else:
    subdiv = QUICK_SUBDIV if MODE == "quick" else SUBDIV
    run = exp.recompute_morse(subdiv=subdiv)
MG_DIR = run.morse_dir
print(f"artifacts -> {MG_DIR}")

### What came out

Each Morse set with its Conley index, how many boxes it occupies, and what it
flows into. A node with no outgoing edges is minimal: an attractor.

In [ ]:
import numpy as np

from latentdynamics.analysis import MorseGraph

graph = MorseGraph.from_dot(MG_DIR / "morse_graph")
boxes = np.atleast_2d(np.loadtxt(MG_DIR / "morse_sets", delimiter=","))
counts = dict(zip(*np.unique(boxes[:, -1].astype(int), return_counts=True)))

print(f"{len(graph.nodes)} Morse sets, {len(graph.minimal)} minimal")
for node in graph.nodes:
    index = graph.labels.get(node, "?").split(":", 1)[-1].strip()
    edges = sorted(graph.edges.get(node, []))
    flow = "minimal" if not edges else "-> " + ", ".join(str(e) for e in edges)
    print(f"  {node}: {index:<16} {counts.get(node, 0):>8d} boxes   {flow}")

## Figures

Rendered from the DOT and CSV above with the paper's palette and axis labels,
so a recomputed run and the paper figure come out of the same code. `BOX_SCALE`
only affects drawing: it inflates Morse sets too small to see.

In [ ]:
from latentdynamics.replay import show_image

figs = run.render_morse(box_scale=BOX_SCALE)
show_image(figs.morse_graph_png, width=600)
for png in (p for p in figs.morse_sets_paths if p.suffix == ".png"):
    show_image(png, width=720)

## Run provenance

In [ ]:
run.diagnostics()